In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from jcopml.pipeline import num_pipe, cat_pipe
from jcopml.utils import save_model, load_model
from jcopml.plot import plot_missing_value
from jcopml.feature_importance import mean_score_decrease

In [2]:
df = pd.read_csv("data/profile.csv")
df.head()

,id,name,city,status,pekerjaan,gaji,kepuasaan_hidup
0,1,Qori Suryono,Lhokseumawe,Berkeluarga,fisherman,3 Juta,76%
1,2,"Darijan Mandala, S.Psi",Tebingtinggi,Single,chef,82 Juta,50%
2,3,Cinta Padmasari,Banjarbaru,Berkeluarga,etc,69 Juta,19%
3,4,Rahmi Pertiwi,Langsa,Berkeluarga,fisherman,99 Juta,34%
4,5,Among Mulyani,Bitung,Single,etc,39 Juta,43%


In [3]:
X = df.drop(columns=["id", "name","city", "status"])
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((7999, 3), (2000, 3), (7999,), (2000,))

In [4]:
preprocessor = ColumnTransformer([
    ('numeric', num_pipe(), []),
    ('categoric', cat_pipe(encoder='onehot'), [ "pekerjaan", "gaji", "kepuasaan_hidup"]),
])

In [5]:
from sklearn.ensemble import RandomForestClassifier
pipeline = Pipeline([
    ('prep', preprocessor),
    ('algo', RandomForestClassifier(n_jobs=-1, random_state=42))
])

In [6]:
from sklearn.model_selection import RandomizedSearchCV
from jcopml.tuning import random_search_params as rsp

model = RandomizedSearchCV(pipeline, rsp.rf_params, cv=3, scoring="accuracy", n_iter=50, n_jobs=-1, verbose=1, random_state=42)
model.fit(X_train, y_train)

print(model.best_params_)
print(model.score(X_train, y_train), model.best_score_, model.score(X_test, y_test))

Fitting 3 folds for each of 50 candidates, totalling 150 fits
{'algo__max_depth': 47, 'algo__max_features': np.float64(0.29659639349515027), 'algo__min_samples_leaf': 20, 'algo__n_estimators': 129}
0.5781972746593325 0.5055635393662814 0.5075
